# Model D: fuzzy uncertainty + causal transformer + IQL

Phase 4 — test of the central hypothesis: making the agent's uncertainty about market
regime EXPLICIT (a FIXED interval type-2 fuzzy layer over vol / momentum / RSI) before
the policy network sees the state should beat exposure-matched control (EM) and the
Phase-2/3 baselines (B: naive_new DDR, C: TACR). Policy = Implicit Q-Learning (IQL,
Kostrikov et al. 2022) over the four Phase-1 behavior-policy trajectories.

Two variants share one package: **D** (`fuzzy=True`, 8 raw + 18 fuzzy = **26-d**) and
**D-minus-fuzzy** (`fuzzy=False`, **8-d**, no padding) — the ablation. All
hyperparameters are identical; the input projection width is the only structural
difference. Pre-registered rules in PROJECT_NOTES 7.8 / spec section 0.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from src.models.d.config import DConfig
from src.models.d.data import load_d_data, split_d_data
from src.models.d.train import train_d, load_agent, roll_split, BEST_NAME, FINAL_NAME
from src.models.d.eval import roll_test_preds, VARIANT_DIRS

SEEDS = [20260814, 1, 2, 3, 4]
cfg = DConfig.from_yaml()  # configs/model_d.yaml (IQL paper values + flagged deviations)
print(cfg)

**Fuzzy layer (fixed, train-split init only).** Centers at the 33rd/50th/66th
percentiles of the TRAIN-split normalized values; UMF std = 0.5 x bin width; LMF std =
0.8 x UMF. Buffers only — no learned parameters, so the treatment is an input-only
transformation.

In [ ]:
# inspect the fixed fuzzy memberships for D (one field, e.g. realized_vol_20d)
import numpy as np
cfg_d = DConfig(fuzzy=True)
data = load_d_data(cfg_d)
print("input dim D =", data.states_in.shape[-1], "| fuzzy n =", data.fuzzy.n_fuzzy_features)
print("centers (realized_vol_20d / ret_20d / rsi_14):")
print(data.fuzzy.centers.numpy().round(3))
print("UMF std:", data.fuzzy.std_umf.numpy().round(3))

**Single-seed run for inspection** (the CLI trains all 5 seeds x both variants; seeds
train independently, so a re-run here is equivalent for the inspected seed).

In [ ]:
from dataclasses import replace
# D (fuzzy, 26-d) — single seed at the screen budget (3k steps)
cfg.seed = SEEDS[0]
cfg.fuzzy = True
train_cfg = replace(cfg, checkpoint_dir=cfg.checkpoint_dir / "d" / f"s{cfg.seed}")
model, history, best_path = train_d(train_cfg)
history

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, ["v_loss", "q_loss", "pi_loss", "val_sharpe"]):
    ax.plot(history[col]); ax.set_title(col)
fig.tight_layout()

**Basin screening** (protocol carried from B/C): a good-basin run peaks early in the
schedule (Model B: best-val epoch 3-5/30); an anomalously late best-val epoch or low
cross-seed test correlation (< 0.7, calibrated to D's own pack) flags a suspect basin.

In [ ]:
log = history
best_ep = int(log["val_sharpe"].idxmax()) + 1
print(f"seed {cfg.seed}: best-val epoch {best_ep}/{len(log)}")
preds = roll_test_preds(cfg, "D", cfg.seed, BEST_NAME)
print(preds.head())

**Reading the regime table** — per-regime Sharpe is the PRIMARY result; `all` (blended)
is secondary. `crisis` rests on 15 test days — directionally suggestive only, never a
headline. The CLI aggregates all 5 seeds x both variants and applies the pre-registered
screen / escalation / ablation rules (see checkpoints/d/screen_summary.csv).

In [ ]:
# D-minus-fuzzy (8-d) — same protocol; the only difference is the input width
cfg_mf = DConfig(fuzzy=False)
cfg_mf.seed = SEEDS[0]
train_cfg_mf = replace(cfg_mf, checkpoint_dir=cfg_mf.checkpoint_dir / "d_minus_fuzzy" / f"s{cfg_mf.seed}")
model_mf, history_mf, _ = train_d(train_cfg_mf)
print(history_mf)

**After the full CLI run** (`python scripts/d_model_run.py`) every artifact lives under
`src/models/d/checkpoints/`: `d/` and `d_minus_fuzzy/` hold per-seed `d_best.pt` +
`d_final.pt` (final-epoch sanity check), `regime_eval.csv`, `basin_screening.csv`, and
`vs_model_b.csv`; `screen_summary.csv` records the verdicts. If the screen triggered, the
20k escalation run is under `checkpoints/d_20k/`.